# Lab 7 - Diagnosing slow visuals (browser only)

You need nothing but this notebook. We use **semantic-link** (already installed
in Fabric) to trace each query and read the **storage engine (SE) vs formula
engine (FE)** split - the same information DAX Studio's Server Timings shows.

**The workflow:** run three slow queries, read the split, classify each as
SE-bound or FE-bound, then apply the cheapest fix and watch the split move.

## 1. Point at your model
Set these to the model you copied into **your own** workspace during bootstrap.

In [ ]:
# EDIT THESE TWO LINES
WORKSPACE = "My Workspace"            # <- your own workspace name
MODEL     = "07 Slow Visual Triage"  # <- your semantic model name

## 2. The measurement helper
Run this once. `measure(dax, label)` traces a query and prints the SE/FE split.

In [ ]:
# Server Timings, in a notebook. semantic-link (sempy) is already installed in
# Fabric, so there is nothing to install for this cell.
import sempy.fabric as fabric
import pandas as pd
import time

# The events we want. QueryEnd = the whole query; VertiPaqSEQueryEnd = storage
# engine (SE) scans. Formula engine (FE) time is the remainder.
EVENTS = {
    "QueryEnd":                  ["EventClass", "EventSubclass", "TextData", "Duration", "CpuTime"],
    "VertiPaqSEQueryEnd":        ["EventClass", "EventSubclass", "TextData", "Duration", "CpuTime"],
    "VertiPaqSEQueryCacheMatch": ["EventClass", "TextData"],
}

def _col(df, name):
    """Find a column ignoring spaces/case (sempy names vary by version)."""
    key = name.replace(" ", "").lower()
    for c in df.columns:
        if c.replace(" ", "").lower() == key:
            return c
    return None

def measure(dax, label, settle=5, show_raw=False):
    """Run a DAX query, trace it, and return a total / SE / FE breakdown in ms."""
    with fabric.create_trace_connection(dataset=MODEL, workspace=WORKSPACE) as tc:
        with tc.create_trace(EVENTS, "Perf trace") as tr:
            tr.start()
            fabric.evaluate_dax(dataset=MODEL, dax_string=dax, workspace=WORKSPACE)
            time.sleep(settle)                 # let the trace events flush
            logs = tr.stop()

    if show_raw or _col(logs, "EventClass") is None:
        display(logs)                          # fall back to the raw trace

    ec, dur = _col(logs, "EventClass"), _col(logs, "Duration")
    total = float(logs.loc[logs[ec] == "QueryEnd", dur].max() or 0)
    se    = float(logs.loc[logs[ec] == "VertiPaqSEQueryEnd", dur].sum() or 0)
    fe    = max(total - se, 0.0)
    bound = "SE-bound" if total and se / total >= 0.5 else "FE-bound"
    print(f"{label:<22}  total {total:>7.0f} ms   SE {se:>7.0f} ms   FE {fe:>7.0f} ms   -> {bound}")
    return {"Query": label, "Total ms": total, "SE ms": se, "FE ms": fe, "Bound": bound}

## 3. Measure the three slow visuals

Each query below stands in for one slow visual. Run the cell, then read the
printed split. Do not tune anything yet - diagnose first.

In [ ]:
# Three deliberately slow queries (one per planted problem).
# Replace the measure names/tables with the ones in your model if they differ.
q_a = """
EVALUATE
SUMMARIZECOLUMNS ( 'Date'[MonthYear], "Sales", [Total Sales] )
"""   # Visual A - scans the full fact (suspect: bad model / missing aggregation)

q_b = """
EVALUATE
SUMMARIZECOLUMNS (
    'Product'[Category],
    "Heavy", [Deliberately Expensive Measure]
)
"""   # Visual B - expensive measure logic (suspect: FE-bound)

q_c = """
EVALUATE
SUMMARIZECOLUMNS ( 'Sales'[OrderNumber], "Rows", COUNTROWS ( 'Sales' ) )
"""   # Visual C - high-cardinality grouping (suspect: SE-bound)

results = [measure(q_a, "Visual A"), measure(q_b, "Visual B"), measure(q_c, "Visual C")]
summary = pd.DataFrame(results)
display(summary)

## 4. Read the split, then fix the cheapest thing

- **SE-bound** (SE is most of the time): you are scanning too much. Fix the
  model, add an aggregation, or cut cardinality.
- **FE-bound** (FE is most of the time): the measure logic is expensive.
  Simplify the DAX.

Match each visual to its root cause, apply one fix in your model (web editing),
then re-run the relevant query below to confirm the split moved as you expected.

In [ ]:
# After you fix ONE visual in the model, re-measure just that query:
display(pd.DataFrame([measure(q_a, "Visual A (after fix)")]))

## Optional - if you happen to have DAX Studio

The same three queries are in `lab07-slow-visuals.dax`. Paste them into DAX
Studio with **Server Timings** on to see the identical SE/FE breakdown. This is
completely optional; the notebook above already gives you everything you need.